<a href="https://colab.research.google.com/github/byeongjun-03/kfa-ai-agent/blob/main/w06_%EC%B4%88%EC%95%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼 2조 초안 노트북: 내 산출물 한 장 뽑기

4회차 검증기와 같은 엔진으로 **내 종목의 초안 페이지(HTML 한 장)** 를 만들어요.

```
DART 여러 해·분기 ─┐
내 소스(3회차)  ───┼─▶ rid · [S#] ─▶ 지표 [M#] ─▶ Gemini가 내 문장의 근거 찾기 ─▶ 숫자 대조 ─▶ 초안 페이지
손으로 옮긴 숫자 ──┘                                                     (PASS / FIX / REVIEW)
```

- **문장은 내가 써요.** AI는 내 문장 속 숫자가 표의 어느 줄에서 왔는지 고르기만 하고, 맞는지는 **코드가** 판정해요.
- 페이지는 **리포트형**(투자포인트 → 실적 표 → 내 핵심 표 → 차트)과 **지표분석형**(결론 → 지표 표·판정 → 차트 → 내 소스) 두 가지예요.

**사용법:** 아래 설정 셀을 내 것으로 바꾸고 위 메뉴 **런타임 → 모두 실행**. "보안 비밀 액세스 허용" 창이 뜨면 **허용**.
처음 한 번은 설정을 바꾸지 말고 그대로 돌려 보세요. SK하이닉스 예시 페이지가 끝까지 나오면 준비 끝이에요.

**설정**

| 이름 | 무엇 | 예 |
|---|---|---|
| `STOCK_CODE` | 종목코드 6자리. 따옴표 안에 앞의 0까지 | `"005930"` |
| `KIND` | `"리포트형"` 또는 `"지표분석형"` | `"지표분석형"` |
| `TITLE` | 페이지 맨 위 제목 | `"삼성전자 3Q26 실적 리포트 초안"` |
| `AUTHOR` | 작성자 이름 | `"하경민"` |
| `YEARS` | 사업보고서(연간) 연도 목록. 각 해의 **전년도도 자동으로** 받아서 성장률을 계산해요 | `[2024, 2025]` |
| `QUARTERS` | 분기 목록 `(연도, 분기)`. 분기는 1~3만(4분기는 `YEARS`에 연도로). 1년 전 같은 분기도 자동으로 받아요 | `[(2026, 2), (2026, 3)]` |
| `TEXT` | **내가 쓴 문장** (리포트형은 투자포인트, 지표분석형은 결론). 빈 줄로 문단을 나눠요 | |
| `MY_TABLE_TITLE` | 내 소스·손으로 옮긴 숫자 표의 제목 | `"분기말 원/달러 환율"` |
| `CHART_ITEMS` | 차트로 그릴 항목 이름 목록. 비우면 매출액·영업이익 막대 차트 | `["종가"]` |
| `CHART_REBASE` | `True`면 첫날을 100으로 맞춰 그려요. 단위가 다른 항목(주가와 지수)을 한 차트에 그릴 때 | `True` |
| `CAVEATS` | 페이지 '확인 필요'에 내가 직접 남길 메모 | `["부문별 매출은 11/30에 추가"]` |

기본값은 4회차 기본 글과 똑같아서, Gemini 키가 없어도 스터디장이 미리 받아 둔 답으로 끝까지 돌아가요.

In [25]:
STOCK_CODE = "005380"
KIND = "리포트형"
TITLE = "현대차: 실적보다 환율이 향후의 리스크"
AUTHOR = "병준"
YEARS = [2025]
QUARTERS = [(2026, 1), (2026, 2), (2026, 3)]
TEXT = """
현대차의 2025년 매출은 186.3조 원으로 전년보다 6.3% 늘었다. 2025년 영업이익률은 6.2%였다. 부채비율은 189% 수준이다.

2026년 3분기 영업이익은 2.7조 원이었다. 원/달러 기말환율은 2026년 6월 말 1,541.5원에서 9월 말 1,358.4원으로 떨어졌다.
"""
MY_TABLE_TITLE = "원/달러 기말환율"
CHART_ITEMS = ["원/달러 기말환율"]
CHART_REBASE = False
CAVEATS = ["지역별 도매판매(현대차 IR 실적발표)는 11/30에 추가"]

In [26]:
#@title 📦 설치 (1분) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "finance-datareader", "pykrx", "python-dotenv", "google-genai"],
                      capture_output=True, text=True)
print("✅ 설치 완료" if proc.returncode == 0 else "❌ 설치 실패: 런타임 → 런타임 다시 시작 → 모두 실행")

✅ 설치 완료


In [27]:
#@title 🧰 공통 도구 (고치지 말고 실행만. 코드가 궁금하면 이 셀을 더블클릭) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
REPO_RAW_URL = "https://raw.githubusercontent.com/phm0305/KFA-ai-agent-study/main/"
DART_CORPS_FILE = "backup/dart_corps.csv"
_DART_CORPS = {}
_DART_CORPS_LOADED = set()


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


def download_dart_corps(api_key):
    """Listed companies from DART's full corpCode.xml: {stock_code: (corp_code, corp_name)}."""
    resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key}, timeout=600)
    try:
        xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
    except zipfile.BadZipFile:
        try:
            _dart_status(resp.json())
        except ValueError:
            pass
        raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
    corps = {}
    for item in ET.fromstring(xml_bytes).iter("list"):
        code = (item.findtext("stock_code") or "").strip()
        if code:
            corps[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    return corps


def read_dart_corps(content):
    # dtype=str keeps leading zeros: read as numbers, "000660" would become 660 and never match.
    table = pd.read_csv(io.BytesIO(content), dtype=str, keep_default_na=False)
    return {row.stock_code: (row.corp_code, row.corp_name) for row in table.itertuples(index=False)}


# [1단계] 종목코드 → DART 회사 번호. 레포의 작은 상장사 표를 먼저 보고, 없을 때만 DART 전체 목록을 받음
def dart_corp(stock_code, api_key):
    if "repo" not in _DART_CORPS_LOADED:
        _DART_CORPS_LOADED.add("repo")
        try:
            _DART_CORPS.update(read_dart_corps(http_get(REPO_RAW_URL + DART_CORPS_FILE, timeout=10).content))
        except CollectError:
            pass
    if stock_code not in _DART_CORPS and "dart" not in _DART_CORPS_LOADED:
        print("DART 회사 목록을 받는 중이에요. 처음 한 번, 몇 분 걸릴 수 있어요")
        _DART_CORPS.update(download_dart_corps(api_key))
        _DART_CORPS_LOADED.add("dart")
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        memo = base_note
        if is_flow and quarter == 4:
            memo += "; 4분기 3개월치 = 연간 - 3분기 누적"
        elif is_flow and quarter == 1:
            memo += "; 1분기는 3개월 = 누적"
        amounts = (row.get("thstrm_amount"), row.get("thstrm_add_amount"))
        for label, amount in zip(_dart_labels(name, is_flow, quarter), amounts):
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, is_flow) in DART_ITEMS.items():
        if account not in seen:
            for label in _dart_labels(name, is_flow, quarter):
                out.append({"항목": label, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                            "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


def _dart_labels(name, is_flow, quarter):
    """Row names for one account; found and missing accounts share them so metrics find both."""
    if not is_flow:
        return [name]
    if quarter == 4:
        return [f"{name}(연간)"]
    if quarter == 1:
        return [f"{name}(3개월)"]
    return [f"{name}(3개월)", f"{name}(누적)"]


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구 준비 완료")

✅ 공통 도구 준비 완료


In [28]:
#@title ⚙️ 엔진 도구 (고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: report engine skeleton.

Adds row ids and a numbered source list on top of the shared record table from
collect.py, computes metrics from those rows, and checks the numbers written in
claims against them. Engine columns go on copies only; the 15 shared columns never
change. Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    with_ids()        공통 표의 줄마다 이름표(rid)를 붙여요. 같은 사실이 두 줄이면 멈춰요
    build_sources()   출처 목록 [S1] [S2] ...을 만들고, 줄마다 어느 출처인지(source_id) 붙여요
    compute_metrics() 지표 정의 표(METRIC_SPECS)대로 지표 [M1] [M2] ...을 계산해요. 입력 줄이 없으면 값을 비우고 이유를 적어요
    check_claims()    주장에 적은 숫자를 지표나 원본 줄과 대조해서 PASS / FIX / REVIEW로 판정해요
    render_check()    주장 목록·지표 표·숫자 대조 표·출처 목록을 마크다운 파일 하나로 저장해요
    claims_prompt()   Gemini에게 보낼 질문(규칙 + 공통 표 + 지표 표 + 글)을 만들어요
    ask_claims()      Gemini 답을 원문 그대로 저장해요. 질문이 저장된 것과 똑같으면 다시 부르지 않아요
    use_backup()      Gemini가 안 될 때, 질문이 똑같으면 스터디장이 미리 받아 둔 답(backup/)을 써요
    load_claims()     저장된 답을 주장 목록으로 읽어요. 깨진 주장은 REVIEW로 넘겨요
"""

import json
import re
import time
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import pandas as pd


RID_PARTS = ["종목코드", "항목", "기준", "기준일", "출처"]
SOURCE_KEY = ["출처", "출처 URL"]
SOURCE_COLUMNS = ["source_id", "출처", "출처 URL", "수집 방법", "신뢰도", "기준일 범위", "줄 수"]
METRIC_COLUMNS = ["metric_id", "지표", "값", "단위", "계산식", "입력 rid", "source_id", "기준일", "상태", "이유"]
CHECK_COLUMNS = ["claim_id", "문장", "근거 id", "적은 숫자", "적은 단위", "실제 값", "판정", "이유", "출처 번호"]
METRIC_BASIS = "연결"
COMPUTED = "계산됨"
PASS, FIX, CHECK = "PASS", "FIX", "REVIEW"
UNITS = {
    "원": ("원", Decimal(1)),
    "천원": ("원", Decimal(10) ** 3),
    "백만원": ("원", Decimal(10) ** 6),
    "억원": ("원", Decimal(10) ** 8),
    "십억원": ("원", Decimal(10) ** 9),
    "조원": ("원", Decimal(10) ** 12),
    "%": ("%", Decimal(1)),
}


@dataclass(frozen=True)
class MetricSpec:
    """growth: inputs = (item,), compared with the same item one year earlier.
    ratio: inputs = (numerator item, denominator item), same date."""
    name: str
    unit: str
    formula: str
    kind: Literal["growth", "ratio"]
    inputs: tuple


METRIC_SPECS = (
    MetricSpec("매출 성장률", "%", "(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100",
               "growth", ("매출액(연간)",)),
    MetricSpec("영업이익률", "%", "영업이익(연간) ÷ 매출액(연간) × 100", "ratio", ("영업이익(연간)", "매출액(연간)")),
    MetricSpec("부채비율", "%", "부채총계 ÷ 자본총계 × 100", "ratio", ("부채총계", "자본총계")),
)


def _text(value):
    return "" if is_blank(value) else str(value)


def _source_order(sid):
    return int(sid[1:])


def _require_tagged(records):
    missing = [col for col in ("rid", "source_id") if col not in records.columns]
    if missing:
        raise CollectError(f"공통 표에 {missing} 칸이 없어요. build_sources(with_ids(표)) 결과를 넣어 주세요")


def with_ids(records):
    """Copy with rid = 종목코드|항목|기준|기준일|출처; a repeated rid is the same fact twice and stops."""
    out = records.copy()
    out["rid"] = ["|".join(_text(v) for v in row) for row in out[RID_PARTS].itertuples(index=False)]
    repeated = out.loc[out["rid"].duplicated(keep=False), "rid"].unique().tolist()
    if repeated:
        raise CollectError(
            f"같은 사실(종목코드|항목|기준|기준일|출처)이 두 줄 이상 있어요: {repeated[:5]}. "
            "같은 자료를 두 번 붙였는지 확인해 주세요")
    return out


def build_sources(records):
    """Return (sources, records copy with source_id).

    One source = one (출처, 출처 URL) pair, so each DART filing keeps its own URL.
    Numbers follow the sorted pairs, not row order, so reordering cells keeps every S#.
    """
    out = records.copy()
    keys = [tuple(_text(v) for v in row) for row in out[SOURCE_KEY].itertuples(index=False)]
    ids = {key: f"S{n}" for n, key in enumerate(sorted(set(keys)), 1)}
    out["source_id"] = [ids[key] for key in keys]
    rows = []
    for (name, url), sid in ids.items():
        part = out[out["source_id"] == sid]
        row = {"source_id": sid, "출처": name, "출처 URL": url, "줄 수": len(part)}
        for col in ("수집 방법", "신뢰도"):
            values = part[col].map(_text).unique().tolist()
            if len(values) > 1:
                raise CollectError(f"[{sid}] {name}: 한 출처인데 {col}이(가) 여러 개예요 {values}. 하나로 맞춰 주세요")
            row[col] = values[0]
        dates = sorted(d for d in part["기준일"].map(_text) if d)
        if dates:
            row["기준일 범위"] = dates[0] if dates[0] == dates[-1] else f"{dates[0]} ~ {dates[-1]}"
        else:
            row["기준일 범위"] = ""
        rows.append(row)
    return pd.DataFrame(rows, columns=SOURCE_COLUMNS), out


def _find_input(records, stock_code, item, base_date):
    """Return (row, None) for exactly one usable consolidated row, else (row or None, reason)."""
    label = f"{item} {base_date}"
    same = records[(records["종목코드"].map(_text) == stock_code) & (records["항목"].map(_text) == item)
                   & (records["기준일"].map(_text) == base_date)]
    rows = same[same["기준"].map(_text) == METRIC_BASIS]
    if rows.empty:
        hint = " (다른 기준 줄만 있음)" if not same.empty else ""
        return None, f"{label}: 연결 줄 없음{hint}"
    if len(rows) > 1:
        return None, f"{label}: 연결 줄이 {len(rows)}개라 하나를 고를 수 없음 ({rows['출처'].tolist()})"
    row = rows.iloc[0]
    if row["상태"] == REVIEW:
        return row, f"{label}: 입력 줄이 확인 필요 ({_text(row['비고'])})"
    if is_blank(row["값"]):
        return row, f"{label}: 값 없음"
    return row, None


def compute_metrics(records, specs, stock_code, base_date):
    """One row per spec, M# in spec order. A metric whose inputs are missing, flagged,
    ambiguous or have a denominator <= 0 keeps an empty value with 확인 필요 and the reasons."""
    _require_tagged(records)
    prev_date = f"{int(base_date[:4]) - 1}{base_date[4:]}"
    out = []
    for n, spec in enumerate(specs, 1):
        if spec.kind == "growth":
            wanted = [(spec.inputs[0], base_date), (spec.inputs[0], prev_date)]
        else:
            wanted = [(item, base_date) for item in spec.inputs]
        found, reasons = [], []
        for item, date in wanted:
            row, reason = _find_input(records, stock_code, item, date)
            if row is not None:
                found.append(row)
            if reason:
                reasons.append(reason)
        value = None
        if not reasons:
            first, second = (float(row["값"]) for row in found)
            if second <= 0:
                label = f"{wanted[1][0]} {wanted[1][1]}"
                if spec.kind == "growth":
                    reasons.append(f"기준값 {label}이(가) 0 이하(적자): 흑자전환 / 적자지속 / 적자확대로 표기")
                else:
                    reasons.append(f"분모 {label}이(가) 0 이하라 비율이 의미 없음")
            else:
                value = ((first - second) if spec.kind == "growth" else first) / second * 100
        out.append({
            "metric_id": f"M{n}", "지표": spec.name, "값": value, "단위": spec.unit, "계산식": spec.formula,
            "입력 rid": [row["rid"] for row in found],
            "source_id": sorted({row["source_id"] for row in found}, key=_source_order),
            "기준일": base_date, "상태": REVIEW if reasons else COMPUTED, "이유": "; ".join(reasons),
        })
    metrics = pd.DataFrame(out, columns=METRIC_COLUMNS)
    metrics["값"] = pd.Series([row["값"] for row in out], dtype=object)
    return metrics


def _written_number(number):
    """Exact decimal of the number as written, so '46.80' keeps its two decimals."""
    if isinstance(number, bool):
        return None
    try:
        value = Decimal(str(number).strip().replace(",", ""))
    except InvalidOperation:
        return None
    return value if value.is_finite() else None


def _unit_scales(unit, actual_unit):
    """Return ((written scale, actual scale), None) or (None, reason).

    Identical unit text compares 1:1 whatever the unit is, because the rid or M# already
    fixes what is measured. Blank units are refused, or two blanks would match as equal text.
    """
    written, source = "".join(_text(unit).split()), "".join(_text(actual_unit).split())
    if not written or not source:
        return None, f"단위 없음 (문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}')"
    if written == source:
        return (Decimal(1), Decimal(1)), None
    if written in UNITS and source in UNITS and UNITS[written][0] == UNITS[source][0]:
        return (UNITS[written][1], UNITS[source][1]), None
    return None, (f"단위가 안 맞음: 문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}' "
                  "(단위 글자가 같거나, 원·천원·백만원·억원·십억원·조원끼리만 바꿔 비교)")


def _check_one(number, unit, actual, actual_unit):
    """Return (actual in the written unit or None, verdict, reason)."""
    written = _written_number(number)
    if written is None:
        return None, CHECK, f"적은 숫자 '{number}'를 숫자로 못 읽음"
    scales, problem = _unit_scales(unit, actual_unit)
    if problem:
        return None, CHECK, problem
    scale, actual_scale = scales
    actual_base = Decimal(str(actual)) * actual_scale
    tolerance = Decimal("0.5").scaleb(written.as_tuple().exponent) * scale
    shown = float(actual_base / scale)
    if abs(written * scale - actual_base) <= tolerance:
        return shown, PASS, ""
    return shown, FIX, f"실제 값은 {_format_actual(shown, written, unit)}"


def _format_actual(actual, written, unit):
    """Two more decimals than the written number, so a FIX shows how far off it is."""
    places = max(-written.as_tuple().exponent, 0) + 2
    return f"{actual:,.{places}f} {unit}"


def _lookup(evidence_id, metrics, records):
    """Return (value, unit, source ids, reason the value cannot be used or None)."""
    if not evidence_id:
        return None, None, [], "근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음)"
    if evidence_id.startswith("M"):
        hit = metrics[metrics["metric_id"] == evidence_id]
        if hit.empty:
            return None, None, [], f"지표 {evidence_id}를 못 찾음"
        row = hit.iloc[0]
        if row["상태"] == REVIEW:
            return None, None, list(row["source_id"]), f"근거 지표가 확인 필요: {row['이유']}"
        return row["값"], row["단위"], list(row["source_id"]), None
    hit = records[records["rid"] == evidence_id]
    if hit.empty:
        return None, None, [], "근거 rid를 공통 표에서 못 찾음"
    row = hit.iloc[0]
    if row["상태"] == REVIEW:
        return None, None, [row["source_id"]], "근거 줄이 확인 필요"
    if is_blank(row["값"]):
        return None, None, [row["source_id"]], "근거 줄에 값 없음"
    return row["값"], _text(row["단위"]), [row["source_id"]], None


RID_FILLED = "rid 보완"


def _complete_rid(evidence_id, records):
    """Return (rid to look up, fill note or None, problem or None).

    A model sometimes cites a rid without its last field (the source). Such a four-field id
    is completed only when exactly one row shares those four fields, so it never picks a source.
    """
    if not evidence_id or evidence_id.startswith("M") or evidence_id.count("|") != 3:
        return evidence_id, None, None
    if (records["rid"] == evidence_id).any():
        return evidence_id, None, None
    hits = records.loc[records["rid"].str.rsplit("|", n=1).str[0] == evidence_id, "rid"]
    if len(hits) == 1:
        return hits.iloc[0], f"{RID_FILLED} (출처 칸이 빠진 rid를 {hits.iloc[0]}로 읽음)", None
    if len(hits) > 1:
        return evidence_id, None, f"근거 rid에 출처 칸이 없고 맞는 줄이 {len(hits)}개라 고를 수 없음"
    return evidence_id, None, None


def check_claims(claims, metrics, records):
    """One row per (claim, evidence). Evidence ids starting with M are metrics, others are rids.

    claims: [{"claim_id": "C-1", "문장": "...", "근거": [(id, number, unit), ...]}].
    The number may be text ("46.80", "1,100") so trailing zeros keep their precision.
    A claim carrying "형식 오류" (set by load_claims) yields one REVIEW row and nothing else.
    출처 번호 collects the S# of every evidence row, or of a metric's input rows.
    """
    _require_tagged(records)
    out = []
    for claim in claims:
        evidence = claim.get("근거") or []
        rows, sources = [], set()
        problem = claim.get("형식 오류")
        if problem or not evidence:
            rows.append({"근거 id": "", "적은 숫자": None, "적은 단위": "", "실제 값": None, "판정": CHECK,
                         "이유": f"AI 답변 형식 오류: {problem}" if problem else "근거 없음"})
            evidence = []
        for evidence_id, number, unit in evidence:
            rid, filled, problem = _complete_rid(evidence_id, records)
            if not problem:
                actual, actual_unit, sids, problem = _lookup(rid, metrics, records)
                sources.update(sids)
            if problem:
                shown, verdict, reason = None, CHECK, problem
            else:
                shown, verdict, reason = _check_one(number, unit, actual, actual_unit)
            if filled:
                reason = f"{filled}; {reason}" if reason else filled
            rows.append({"근거 id": evidence_id, "적은 숫자": number, "적은 단위": unit, "실제 값": shown,
                         "판정": verdict, "이유": reason})
        label = "".join(f"[{sid}]" for sid in sorted(sources, key=_source_order))
        for row in rows:
            out.append({"claim_id": claim.get("claim_id"), "문장": claim.get("문장"), **row, "출처 번호": label})
    return pd.DataFrame(out, columns=CHECK_COLUMNS)


def _cell(value):
    return _text(value).replace("|", "\\|").replace("\n", " ")


def _table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(_cell(v) for v in row) + " |" for row in rows]
    return lines


def _claim_number_rows(checked):
    for _, row in checked.iterrows():
        number, unit, actual = row["적은 숫자"], row["적은 단위"], row["실제 값"]
        written = _written_number(number)
        yield [
            f"{row['claim_id']} {row['문장']}",
            "" if is_blank(number) else f"{number} {unit}",
            "" if is_blank(actual) or written is None else _format_actual(actual, written, unit),
            f"{row['판정']}: {row['이유']}" if row["판정"] == CHECK
            else f"{row['판정']} ({RID_FILLED})" if _text(row["이유"]).startswith(RID_FILLED) else row["판정"],
        ]


def render_check(checked, metrics, sources, path):
    """Write claims, metrics, number checks and sources as one markdown file; return its path."""
    claims = checked.drop_duplicates("claim_id")
    lines = ["# 숫자 대조 결과", "", "## 주장", ""]
    lines += [f"- {row['claim_id']} {row['문장']} {row['출처 번호']}".rstrip() for _, row in claims.iterrows()]
    lines += ["", "## 지표", ""]
    lines += _table(["번호", "지표", "값", "계산식", "출처", "기준일"], [
        [row["metric_id"], row["지표"],
         f"확인 필요: {row['이유']}" if is_blank(row["값"]) else f"{row['값']:,.2f} {row['단위']}",
         row["계산식"], "".join(f"[{sid}]" for sid in row["source_id"]), row["기준일"]]
        for _, row in metrics.iterrows()])
    lines += ["", "## 숫자 대조", ""]
    lines += _table(["주장", "적은 숫자", "실제 값", "판정"], _claim_number_rows(checked))
    lines += ["", "## 출처", ""]
    lines += _table(["번호", "이름", "기준일 범위", "원본 URL", "신뢰도"], [
        [f"[{row['source_id']}]", row["출처"], row["기준일 범위"], row["출처 URL"], row["신뢰도"]]
        for _, row in sources.iterrows()])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


BACKUP_URL = REPO_RAW_URL + "backup/"
RETRY_CODES = (429, 500, 503)
RETRY_WAITS = (5, 10, 20)
CLAIM_RULES = """아래 글에서 숫자가 들어간 주장을 모두 찾아, 각 숫자의 근거를 공통 표나 지표 표에서 골라 줘.

규칙
- 출력은 JSON 배열 하나만: [{"claim_id": "C-1", "문장": "...", "근거": [["근거 id", "숫자", "단위"]]}]
- claim_id는 C-1, C-2 ... 순서대로 붙여
- 문장은 글에 적힌 그대로 옮겨
- 근거 id는 공통 표의 rid나 지표 표의 metric_id(M1 ...)를 글자 그대로 복사해. 표에 없는 id를 만들지 마
- 숫자와 단위는 문장에 적힌 글자 그대로 적어 (예: "97.1", "조 원"). 단위를 바꾸거나 다시 계산하지 마
- 문장 속 숫자마다 근거를 하나씩 적어. 숫자가 가리키는 항목의 줄이나 지표를 골라. 값이 맞는지는 판단하지 마
- 증감, 포인트 차이 같은 변화량은 그 변화량 자체가 표에 있을 때만 고르고, 없으면 근거 id를 ""로 비워 (수준 지표에 변화량을 대지 마)
- 그 밖에는 항목 자체가 표에 없을 때만 근거 id를 ""로 비워
- 숫자가 없는 주장은 "근거": []로 적어"""


def claims_prompt(records, metrics, text):
    """Question text for Gemini: rules, the tables it may cite, then the text to read."""
    rows = records[["rid", "항목", "값", "단위", "기준일"]].to_csv(index=False, lineterminator="\n")
    metric_rows = metrics[["metric_id", "지표", "값", "단위"]].to_csv(index=False, lineterminator="\n")
    return f"{CLAIM_RULES}\n\n[공통 표]\n{rows}\n[지표 표]\n{metric_rows}\n[글]\n{text.strip()}\n"


def _call_gemini(prompt, api_key, model, fallback_model):
    """Return (answer text, model that answered, raw errors of the failed tries).
    Same retry rule as setup/00_setup_check.ipynb: wait 5/10/20 s on 429/500/503,
    then try the fallback model once."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(response_mime_type="application/json", temperature=0)
    errors = []

    def ask(name):
        try:
            return client.models.generate_content(model=name, contents=prompt, config=config).text or ""
        except Exception as error:
            errors.append(f"{name} {getattr(error, 'code', None) or type(error).__name__}: {error}")
            raise

    try:
        for attempt, wait in enumerate((*RETRY_WAITS, None)):
            try:
                return ask(model), model, errors
            except Exception as error:
                code = getattr(error, "code", None)
                if code not in RETRY_CODES:
                    raise
                if wait is None:
                    break
                what = "Gemini 요청 한도에 걸렸어요" if code == 429 else "Gemini가 붐벼요"
                print(f"⏳ {what} ({code}). {wait}초 기다렸다가 다시 시도해요 ({attempt + 1}/{len(RETRY_WAITS)})")
                time.sleep(wait)
        print(f"⏳ 계속 안 돼서 예비 모델({fallback_model})로 한 번 더 시도해요")
        return ask(fallback_model), fallback_model, errors
    except Exception as error:
        raw = errors[-1] if errors else str(error)
        reason = _gemini_failure(getattr(error, "code", None), type(error).__name__)
        raise CollectError(f"{reason}\n   원문: {raw[:300]}") from None


def _gemini_failure(code, error_name):
    """Member-facing reason a Gemini call failed. Only server-side trouble says 'try later';
    a key, quota or model problem names what to do instead."""
    if code in (400, 401, 403):
        return f"Gemini가 키를 받아 주지 않았어요 ({code}). Secrets의 GEMINI_API_KEY 값을 다시 확인해 주세요 (SETUP.md 9번)"
    if code == 404:
        return "Gemini 모델 이름을 찾을 수 없어요 (404). 스터디장에게 알려 주세요"
    if code == 429:
        return "Gemini 무료 요청 한도를 넘었어요 (429). 1분쯤 뒤 다시 해 보고, 계속 나오면 하루 한도일 수 있어요. 아래 원문을 확인하세요"
    if code in RETRY_CODES:
        return f"Gemini 서버가 계속 붐비거나 오류가 나요 ({code}). 내 설정 문제가 아니에요. 잠시 뒤 다시 실행해 주세요"
    return f"Gemini를 부르지 못했어요 ({code or error_name}). 인터넷 연결을 확인하고 잠시 뒤 다시 실행해 주세요"


def _record_path(path):
    return path.with_name(f"{path.stem}_기록.json")


def ask_claims(prompt, api_key, path, model, fallback_model, overwrite=False):
    """Save Gemini's raw answer at path and a record (question, model, time) beside it; return path.

    The saved answer is reused only when the saved question is identical to this one,
    so editing the text never checks an old answer. The record is removed before the
    answer is rewritten, so a record on disk always belongs to the answer next to it.
    """
    path = Path(path)
    record_path = _record_path(path)
    if not overwrite and path.exists() and record_path.exists():
        try:
            saved = json.loads(record_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            saved = {}
        if saved.get("질문") == prompt:
            print(f"♻ 같은 질문의 저장된 답을 다시 써요 ({saved.get('모델')}, {saved.get('호출 시각')})")
            return path
    if not api_key:
        raise SkipStep("Gemini 키가 없어요. Secrets에 GEMINI_API_KEY를 등록하고 노트북 액세스를 켠 뒤 다시 실행해 주세요 (SETUP.md 9번)")
    answer, used_model, errors = _call_gemini(prompt, api_key, model, fallback_model)
    path.parent.mkdir(parents=True, exist_ok=True)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record = {"질문": prompt, "모델": used_model, "호출 시각": now_kst(), "오류": errors}
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"✅ Gemini 답 저장: {path} ({used_model})")
    return path


def use_backup(prompt, path, name):
    """Fall back to the answer the study lead saved in backup/<name>.txt when Gemini fails.

    Used only when the backup's question is identical to this one, so a member who changed
    the text or the company never gets an answer to a different question. Returns path or None.
    """
    path = Path(path)
    try:
        record = json.loads(http_get(BACKUP_URL + quote(f"{name}_기록.json")).content.decode("utf-8"))
        if record.get("질문") != prompt:
            print("⏭ 미리 받아 둔 답은 기본 글·기본 종목용이라 지금 질문에는 쓸 수 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
            return None
        answer = http_get(BACKUP_URL + quote(f"{name}.txt")).content.decode("utf-8")
    except (CollectError, json.JSONDecodeError):
        print("⏭ 미리 받아 둔 답이 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
        return None
    record_path = _record_path(path)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"🗂 Gemini 호출이 안 돼서 스터디장이 미리 받아 둔 답을 썼어요 ({record.get('모델')}, {record.get('호출 시각')})")
    return path


def _salvage(text):
    """Split a broken answer into claim objects; a piece that does not parse stays a string.
    Claims hold no nested objects, so every '{' starts a new claim."""
    decoder, items = json.JSONDecoder(), []
    start = text.find("{")
    while start != -1:
        try:
            item, end = decoder.raw_decode(text, start)
        except json.JSONDecodeError:
            end = text.find("{", start + 1)
            items.append(text[start:end if end != -1 else len(text)])
            start = end
            continue
        items.append(item)
        start = text.find("{", end)
    return items


def _claim_from_answer(item, n, seen):
    if not isinstance(item, dict):
        found = re.search(r'"claim_id"\s*:\s*"([^"]+)"', item) if isinstance(item, str) else None
        claim_id = found.group(1) if found and found.group(1) not in seen else f"AI-{n}"
        seen.add(claim_id)
        return {"claim_id": claim_id, "문장": "", "근거": [],
                "형식 오류": f"JSON이 깨짐: {_text(item).strip()[:80]}"}
    problems = []
    claim_id = item.get("claim_id")
    if not isinstance(claim_id, str) or not claim_id.strip():
        problems.append("claim_id 없음")
        claim_id = f"AI-{n}"
    elif claim_id in seen:
        problems.append(f"claim_id {claim_id} 중복")
        claim_id = f"{claim_id} (AI-{n})"
    seen.add(claim_id)
    sentence = item.get("문장")
    if not isinstance(sentence, str):
        problems.append("문장 없음")
    evidence = item.get("근거")
    if not isinstance(evidence, list) or any(not isinstance(ev, list) or len(ev) != 3 for ev in evidence):
        problems.append("근거는 [근거 id, 숫자, 단위] 목록이어야 함")
        evidence = []
    claim = {"claim_id": claim_id, "문장": _text(sentence),
             "근거": [(_text(ev_id).strip(), number, _text(unit)) for ev_id, number, unit in evidence]}
    if problems:
        claim["형식 오류"] = "; ".join(problems)
    return claim


def load_claims(path):
    """Read a saved Gemini answer into check_claims input. Never raises on a bad answer:
    each claim that cannot be read becomes a claim carrying "형식 오류"."""
    text = Path(path).read_text(encoding="utf-8")
    body = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text)
    try:
        items = json.loads(body)
        items = items if isinstance(items, list) else [items]
    except json.JSONDecodeError:
        items = _salvage(body)
    seen = set()
    claims = [_claim_from_answer(item, n, seen) for n, item in enumerate(items, 1)]
    return claims or [{"claim_id": "AI-1", "문장": "", "근거": [], "형식 오류": "AI 답변에서 주장을 하나도 못 찾음"}]

print("✅ 엔진 도구 준비 완료")

✅ 엔진 도구 준비 완료


In [29]:
#@title 🖼 2조 초안 도구 (고치지 말고 실행만) { display-mode: "form" }
"""KFA study team 2: one draft page per member, built on the shared engine.

Collects several DART periods, computes the engine's metrics for each period, checks the
member's own sentences with the engine checker, and draws the result as one self-contained
HTML page in one of two layouts. collect.py and engine.py are used unchanged.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    collect_periods()   여러 해·분기의 DART 재무제표를 한 번에 받아요. 한 기간이 실패해도 나머지는 계속 받아요
    manual_records()    손으로 옮긴 숫자(IR 자료, 증권사 화면 등)를 공통 표 줄로 바꿔요
    period_metrics()    기간마다 지표를 계산하고 [M1] [M2] ... 번호를 처음부터 다시 매겨요
    judge_metrics()     지표를 내가 정한 기준(이상/이하)과 비교해서 충족/미달을 코드로 판정해요
    render_draft()      리포트형·지표분석형 틀로 HTML 한 장을 만들어요
    show_draft()        만든 HTML을 화면에 보여 주고 파일로 저장해요
"""

import html
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import pandas as pd


REPORT, INDICATOR = "리포트형", "지표분석형"
DART_SOURCE = "DART 단일회사 전체 재무제표 API"
REPORT_NAMES = {1: "1분기보고서", 2: "반기보고서", 3: "3분기보고서", 4: "사업보고서"}
FLOW_ITEMS = ("매출액", "영업이익", "당기순이익")
STOCK_ITEMS = ("자산총계", "부채총계", "자본총계")
QUARTER_SPECS = (
    MetricSpec("매출 성장률(전년 동기)", "%",
               "(이번 분기 매출액(3개월) − 1년 전 같은 분기 매출액(3개월)) ÷ 1년 전 같은 분기 매출액(3개월) × 100",
               "growth", ("매출액(3개월)",)),
    MetricSpec("영업이익률(분기)", "%", "영업이익(3개월) ÷ 매출액(3개월) × 100", "ratio",
               ("영업이익(3개월)", "매출액(3개월)")),
    MetricSpec("부채비율", "%", "부채총계 ÷ 자본총계 × 100", "ratio", ("부채총계", "자본총계")),
)
# Annual and quarterly specs name the same idea differently; the financial table shows them on one row.
METRIC_ROWS = (
    ("매출 성장률", ("매출 성장률", "매출 성장률(전년 동기)")),
    ("영업이익률", ("영업이익률", "영업이익률(분기)")),
    ("부채비율", ("부채비율",)),
)
DIRECTIONS = ("이상", "이하")
SENTENCE_END = re.compile(r"(?<=[.!?])\s+|\n")
VERIFIED, WRONG, NO_NUMBER, NEEDS_CHECK = "검증됨", "숫자 틀림", "숫자 없는 주장", "확인 필요"
STATUS_CLASS = {VERIFIED: "ok", WRONG: "bad", NO_NUMBER: "plain", NEEDS_CHECK: "warn"}
STATUS_RANK = {VERIFIED: 0, NO_NUMBER: 1, NEEDS_CHECK: 2, WRONG: 3}
PALETTE = ("#2f5d8a", "#c8873a", "#5b8c6b", "#8a5a9e", "#9a9a9a")


@dataclass(frozen=True, order=True)
class Period:
    year: int
    quarter: int

    @property
    def base_date(self):
        return f"{self.year}-{QUARTER_END[self.quarter]}"

    @property
    def label(self):
        return str(self.year) if self.quarter == 4 else f"{self.year} {self.quarter}Q"

    @property
    def annual(self):
        return self.quarter == 4

    def item(self, name):
        if name in STOCK_ITEMS:
            return name
        return f"{name}(연간)" if self.annual else f"{name}(3개월)"


def plan_periods(years, quarters):
    """Return (shown periods, fetched periods). A year-ago period is fetched too, so growth
    has a base; it is shown in the table but gets no metrics of its own."""
    for item in quarters:
        if not (isinstance(item, tuple) and len(item) == 2 and item[1] in (1, 2, 3)):
            raise CollectError(f"QUARTERS에는 (연도, 분기) 묶음만 넣어요. 분기는 1~3이고, 4분기는 YEARS에 연도로 넣어요: {item}")
    shown = sorted({Period(int(y), 4) for y in years} | {Period(int(y), int(q)) for y, q in quarters})
    if not shown:
        raise CollectError("YEARS와 QUARTERS가 둘 다 비어 있어요. 적어도 한 해는 넣어 주세요")
    fetched = sorted(set(shown) | {Period(p.year - 1, p.quarter) for p in shown})
    return shown, fetched


def collect_periods(stock_code, fetched, api_key):
    """DART consolidated statements for every period; a failed period is skipped, not fatal."""
    parts = []
    for period in fetched:
        label = f"DART {period.label} {REPORT_NAMES[period.quarter]}"
        result = run_step(label, 1, lambda p=period: collect_dart(stock_code, p.year, p.quarter, "CFS", api_key))
        if result is not None:
            parts.append(result)
    return pd.concat(parts, ignore_index=True) if parts else None


def manual_records(rows, stock_code, stock_name):
    """Hand-copied numbers as shared-table rows (level 5). A row stays 확인 필요 until its 확인 is 'o'."""
    if not rows:
        return None
    table = pd.DataFrame(rows)
    missing = [col for col in ("항목", "값", "단위", "기준일", "출처", "출처 URL", "신뢰도") if col not in table.columns]
    if missing:
        raise CollectError(f"MANUAL_ROWS에 {missing} 칸이 없어요. 가이드의 예시 모양대로 적어 주세요")
    if "기준" not in table.columns:
        table["기준"] = NOT_APPLICABLE
    return collect_table(table, level=5, disclosed=SAME_AS_DATE, stock_code=stock_code, stock_name=stock_name)


def period_metrics(records, stock_code, shown, extra_specs=()):
    """Engine metrics for each shown period, renumbered M1.. in period order.

    지표 carries the period when more than one period is shown, because the AI prompt only
    sees metric_id, 지표, 값 and 단위 and must tell 2024 from 2025. Extra specs run on annual
    periods only, since member items are rarely reported per quarter."""
    frames = []
    for period in shown:
        specs = (*METRIC_SPECS, *extra_specs) if period.annual else QUARTER_SPECS
        part = compute_metrics(records, specs, stock_code, period.base_date)
        part["지표명"] = part["지표"]
        part["기간"] = period.label
        frames.append(part)
    metrics = pd.concat(frames, ignore_index=True)
    metrics["metric_id"] = [f"M{n}" for n in range(1, len(metrics) + 1)]
    if len(shown) > 1:
        metrics["지표"] = metrics["지표명"] + " (" + metrics["기간"] + ")"
    metrics["값"] = pd.Series(list(metrics["값"]), dtype=object)
    return metrics


def judge_metrics(metrics, benchmarks):
    """Add 비교 기준 and 판정 (충족 / 미달 / 판정 불가) from {지표명: (이상|이하, threshold)}."""
    out = metrics.copy()
    out["비교 기준"], out["판정"] = "", ""
    for name, rule in benchmarks.items():
        if not (isinstance(rule, tuple) and len(rule) == 2 and rule[0] in DIRECTIONS):
            raise CollectError(f"BENCHMARKS['{name}']은 ('이상' 또는 '이하', 숫자) 모양이어야 해요: {rule}")
        direction, threshold = rule
        hit = out["지표명"] == name
        if not hit.any():
            print(f"⚠ 기준을 적은 지표 '{name}'이 지표 표에 없어요. 지표 이름을 글자 그대로 맞춰 주세요")
            continue
        for idx in out.index[hit]:
            value = out.at[idx, "값"]
            out.at[idx, "비교 기준"] = f"{threshold:g} {out.at[idx, '단위']} {direction}"
            if is_blank(value):
                out.at[idx, "판정"] = "판정 불가"
            else:
                ok = value >= threshold if direction == "이상" else value <= threshold
                out.at[idx, "판정"] = "충족" if ok else "미달"
    return out


def claim_status(rows):
    """One status per claim from its evidence rows."""
    reasons = [r for r in rows["이유"] if not is_blank(r)]
    if any(r.startswith("AI 답변 형식 오류") for r in reasons):
        return NEEDS_CHECK, reasons
    if (rows["판정"] == FIX).any():
        return WRONG, reasons
    if (rows["판정"] == PASS).all():
        return VERIFIED, reasons
    if len(rows) == 1 and rows.iloc[0]["이유"] == "근거 없음":
        return NO_NUMBER, []
    return NEEDS_CHECK, reasons


def summarize_claims(checked):
    """[(claim_id, 문장, status, reasons, 출처 번호)] in answer order."""
    if checked is None or checked.empty:
        return []
    out = []
    for claim_id, rows in checked.groupby("claim_id", sort=False):
        status, reasons = claim_status(rows)
        out.append((claim_id, rows.iloc[0]["문장"], status, reasons, rows.iloc[0]["출처 번호"]))
    return out


def _norm(text):
    return re.sub(r"\s+", "", str(text or ""))


def text_with_status(text, claims):
    """Split the member's text into paragraphs of (sentence, worst status or None, S# label).
    A sentence is matched to a claim when one contains the other after removing spaces."""
    paragraphs = []
    for block in re.split(r"\n\s*\n", text.strip()):
        sentences = []
        for sentence in SENTENCE_END.split(block):
            sentence = sentence.strip()
            if not sentence:
                continue
            key = _norm(sentence)
            hits = [c for c in claims if _norm(c[1]) and (_norm(c[1]) in key or key in _norm(c[1]))]
            status = max((c[2] for c in hits), key=STATUS_RANK.get) if hits else None
            label = "".join(dict.fromkeys(s for c in hits for s in re.findall(r"\[S\d+\]", c[4] or "")))
            sentences.append((sentence, status, label))
        if sentences:
            paragraphs.append(sentences)
    return paragraphs


def fmt_value(value, unit):
    """Won amounts of 1억 and more in 억원 (the unit Korean reports use), so a share price stays in
    원; other units keep up to two decimals."""
    if is_blank(value):
        return ""
    value = float(value)
    if unit == "원":
        return f"{value / 1e8:,.0f}억원" if abs(value) >= 1e8 else f"{value:,.0f}원"
    text = f"{value:,.2f}".rstrip("0").rstrip(".")
    return f"{text} {unit}".strip()


def _e(value):
    return html.escape("" if is_blank(value) else str(value))


def _sup(label):
    return f'<sup class="src">{_e(label)}</sup>' if label else ""


def _need(reason):
    return f'<span class="need" title="{_e(reason)}">확인 필요</span>'


def _record(records, stock_code, item, base_date):
    rows = records[(records["종목코드"] == stock_code) & (records["항목"] == item)
                   & (records["기준일"] == base_date) & (records["기준"] == "연결")]
    return rows.iloc[0] if len(rows) == 1 else None


def financial_tables(records, metrics, stock_code, fetched, shown):
    """Annual and quarterly periods in separate tables, so a 3-month column never sits beside a year."""
    groups = [[p for p in fetched if p.annual], [p for p in fetched if not p.annual]]
    return "".join(financial_table(records, metrics, stock_code, g, shown) for g in groups if g)


def financial_table(records, metrics, stock_code, fetched, shown):
    unit = "연결, 연간" if fetched[0].annual else "연결, 분기 3개월"
    head = "".join(f"<th>{_e(p.label)}</th>" for p in fetched)
    body = []
    for name in (*FLOW_ITEMS, *STOCK_ITEMS):
        cells = []
        for period in fetched:
            row = _record(records, stock_code, period.item(name), period.base_date)
            if row is None:
                cells.append(f"<td>{_need('DART 줄 없음')}</td>")
            elif row["상태"] == REVIEW or is_blank(row["값"]):
                cells.append(f"<td>{_need(row['비고'])}</td>")
            else:
                cells.append(f"<td>{_e(fmt_value(row['값'], '원'))}{_sup('[' + row['source_id'] + ']')}</td>")
        body.append(f"<tr><th>{_e(name)}</th>{''.join(cells)}</tr>")
    for label, names in METRIC_ROWS:
        cells = []
        for period in fetched:
            hit = metrics[(metrics["기간"] == period.label) & metrics["지표명"].isin(names)] if period in shown else []
            if len(hit) == 0:
                cells.append('<td class="dim">-</td>')
            elif is_blank(hit.iloc[0]["값"]):
                cells.append(f"<td>{_need(hit.iloc[0]['이유'])}</td>")
            else:
                m = hit.iloc[0]
                cells.append(f"<td>{_e(fmt_value(m['값'], m['단위']))}{_sup('[' + m['metric_id'] + ']')}</td>")
        body.append(f'<tr class="metric"><th>{_e(label)}</th>{"".join(cells)}</tr>')
    return (f'<table class="fin"><thead><tr><th>{unit}</th>{head}</tr></thead>'
            f'<tbody>{"".join(body)}</tbody></table>')


def metric_table(metrics, judged):
    cols = ["번호", "지표", "기간", "값"] + (["비교 기준", "판정"] if judged else []) + ["계산식", "출처"]
    rows = []
    for _, m in metrics.iterrows():
        value = _need(m["이유"]) if is_blank(m["값"]) else _e(fmt_value(m["값"], m["단위"]))
        cells = [_e(m["metric_id"]), _e(m["지표명"]), _e(m["기간"]), f'<span class="nw">{value}</span>']
        if judged:
            verdict = m["판정"]
            css = {"충족": "ok", "미달": "bad", "판정 불가": "warn"}.get(verdict, "")
            cells += [f'<span class="nw">{_e(m["비교 기준"])}</span>',
                      f'<span class="chip {css}">{_e(verdict)}</span>' if verdict else ""]
        cells += [f'<span class="formula">{_e(m["계산식"])}</span>', _e("".join(f"[{s}]" for s in m["source_id"]))]
        rows.append("<tr>" + "".join(f"<td>{c}</td>" for c in cells) + "</tr>")
    head = "".join(f"<th>{c}</th>" for c in cols)
    return f'<table class="grid"><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table>'


def mine_table(mine, latest):
    """Member sources: the latest rows of each (종목, 항목), newest first."""
    if mine is None or mine.empty:
        return '<p class="dim">아직 내 소스·수동 입력 줄이 없어요. 노트북 1-1, 1-2 칸을 채우면 여기에 나와요.</p>'
    rows = []
    for _, part in mine.groupby(["종목코드", "항목"], sort=False, dropna=False):
        part = part.sort_values("기준일", ascending=False).head(latest)
        for _, r in part.iterrows():
            value = _need(r["비고"]) if r["상태"] == REVIEW else _e(fmt_value(r["값"], r["단위"]))
            name = next((v for v in (r["종목명"], r["종목코드"]) if not is_blank(v)), "-")
            rows.append(f"<tr><td>{_e(name)}</td><td>{_e(r['항목'])}</td><td>{_e(r['기준일'])}</td>"
                        f"<td class='num'>{value}</td><td>{_e('[' + r['source_id'] + ']')}</td></tr>")
    return ('<table class="grid"><thead><tr><th>종목</th><th>항목</th><th>기준일</th><th>값</th><th>출처</th></tr>'
            f'</thead><tbody>{"".join(rows)}</tbody></table>')


def _ticks(lo, hi, n=4):
    if hi == lo:
        hi = lo + 1
    return [lo + (hi - lo) * i / n for i in range(n + 1)]


def _tick_text(value, span):
    return f"{value:,.0f}" if span >= 10 else f"{value:,.1f}"


def bar_chart(groups, series, unit, title):
    """Grouped bars: groups = x labels, series = [(name, [value or None per group])]."""
    w, h, left, right, top, bottom = 760, 300, 64, 16, 40, 44
    values = [v for _, vals in series for v in vals if v is not None]
    if not values:
        return ""
    lo, hi = min(0, min(values)), max(0, max(values))
    ticks = _ticks(lo, hi)
    lo, hi = ticks[0], ticks[-1]
    y = lambda v: top + (h - top - bottom) * (hi - v) / (hi - lo)
    slot = (w - left - right) / len(groups)
    bar = min(44, slot * 0.6 / len(series))
    parts = [f'<text x="{left}" y="22" class="ct">{_e(title)} ({_e(unit)})</text>']
    for t in ticks:
        parts.append(f'<line x1="{left}" x2="{w - right}" y1="{y(t):.1f}" y2="{y(t):.1f}" class="gl"/>'
                     f'<text x="{left - 8}" y="{y(t) + 4:.1f}" class="yt">{_tick_text(t, hi - lo)}</text>')
    for gi, g in enumerate(groups):
        cx = left + slot * gi + slot / 2
        parts.append(f'<text x="{cx:.1f}" y="{h - bottom + 18}" class="xt">{_e(g)}</text>')
        for si, (_, vals) in enumerate(series):
            v = vals[gi]
            if v is None:
                continue
            x0 = cx - bar * len(series) / 2 + bar * si
            y0, y1 = sorted((y(0), y(v)))
            parts.append(f'<rect x="{x0:.1f}" y="{y0:.1f}" width="{bar - 3:.1f}" height="{max(y1 - y0, 1):.1f}" '
                         f'fill="{PALETTE[si % len(PALETTE)]}" rx="2"/>')
    parts.append(_legend([name for name, _ in series], w, h))
    return f'<svg viewBox="0 0 {w} {h}" class="chart">{"".join(parts)}</svg>'


def line_chart(series, unit, title):
    """Lines over a shared, sorted date axis: series = [(name, [(date, value)])]."""
    w, h, left, right, top, bottom = 760, 300, 64, 16, 40, 44
    dates = sorted({d for _, pts in series for d, _ in pts})
    values = [v for _, pts in series for _, v in pts]
    if len(dates) < 2 or not values:
        return ""
    ticks = _ticks(min(values), max(values))
    lo, hi = ticks[0], ticks[-1]
    pos = {d: i for i, d in enumerate(dates)}
    x = lambda d: left + (w - left - right) * pos[d] / (len(dates) - 1)
    y = lambda v: top + (h - top - bottom) * (hi - v) / (hi - lo)
    parts = [f'<text x="{left}" y="22" class="ct">{_e(title)} ({_e(unit)})</text>']
    for t in ticks:
        parts.append(f'<line x1="{left}" x2="{w - right}" y1="{y(t):.1f}" y2="{y(t):.1f}" class="gl"/>'
                     f'<text x="{left - 8}" y="{y(t) + 4:.1f}" class="yt">{_tick_text(t, hi - lo)}</text>')
    step = max(1, (len(dates) - 1) // 5)
    marks = dates[::step] if dates[::step][-1] == dates[-1] else [*dates[::step][:-1], dates[-1]]
    for d in marks:
        anchor = "start" if d == dates[0] else "end" if d == dates[-1] else "middle"
        parts.append(f'<text x="{x(d):.1f}" y="{h - bottom + 18}" class="xt" style="text-anchor:{anchor}">{_e(d)}</text>')
    for si, (_, pts) in enumerate(series):
        path = " ".join(f"{'M' if i == 0 else 'L'}{x(d):.1f},{y(v):.1f}" for i, (d, v) in enumerate(sorted(pts)))
        parts.append(f'<path d="{path}" fill="none" stroke="{PALETTE[si % len(PALETTE)]}" stroke-width="2"/>')
    parts.append(_legend([name for name, _ in series], w, h))
    return f'<svg viewBox="0 0 {w} {h}" class="chart">{"".join(parts)}</svg>'


def _legend(names, w, h):
    out, x = [], w - 16
    for i, name in reversed(list(enumerate(names))):
        x -= 14 + 13 * len(name)
        out.append(f'<rect x="{x}" y="12" width="10" height="10" fill="{PALETTE[i % len(PALETTE)]}" rx="2"/>'
                   f'<text x="{x + 14}" y="21" class="lg">{_e(name)}</text>')
    return "".join(out)


def default_chart(records, stock_code, fetched):
    """Revenue and operating profit bars; quarters when any are shown, else years."""
    periods = [p for p in fetched if not p.annual] or fetched
    series = []
    for name in ("매출액", "영업이익"):
        vals = []
        for p in periods:
            row = _record(records, stock_code, p.item(name), p.base_date)
            ok = row is not None and row["상태"] != REVIEW and not is_blank(row["값"])
            vals.append(float(row["값"]) / 1e8 if ok else None)
        series.append((name, vals))
    title = "분기 실적 (3개월)" if not periods[0].annual else "연간 실적"
    return bar_chart([p.label for p in periods], series, "억원", title)


def item_chart(records, items, points, rebase):
    """Lines for the named items from any source. Without rebase all items must share one unit."""
    series, units = [], set()
    for item in items:
        rows = records[(records["항목"] == item) & (records["상태"] != REVIEW)]
        for code, part in rows.groupby("종목코드", sort=False, dropna=False):
            part = part.dropna(subset=["값"]).sort_values("기준일").tail(points)
            if len(part) < 2:
                continue
            pts = [(str(d), float(v)) for d, v in zip(part["기준일"], part["값"])]
            if rebase:
                pts = [(d, v / pts[0][1] * 100) for d, v in pts] if pts[0][1] else []
            label = next((v for v in (part["종목명"].iloc[0], code) if not is_blank(v)), "")
            name = item if rows["종목코드"].nunique(dropna=False) == 1 else f"{item} {label}".strip()
            series.append((name, pts))
            units.add(part["단위"].iloc[0])
    if not series:
        print(f"⚠ 차트로 그릴 줄이 없어요: {items}. 항목 이름이 공통 표와 글자까지 같은지 확인해 주세요")
        return ""
    if not rebase and len(units) > 1:
        print(f"⚠ 단위가 다른 항목을 한 차트에 그릴 수 없어요 {sorted(units)}. CHART_REBASE = True로 하면 첫날=100으로 맞춰 그려요")
        return ""
    unit = "첫날=100" if rebase else units.pop()
    return line_chart(series, unit, " · ".join(items))


def review_list(claims, records, metrics, caveats):
    items = [f"<li><b>작성자 메모</b> {_e(c)}</li>" for c in caveats]
    for claim_id, sentence, status, reasons, _ in claims:
        if status in (WRONG, NEEDS_CHECK):
            items.append(f"<li><b>{_e(claim_id)} {_e(status)}</b> {_e(sentence)} "
                         f'<span class="dim">({_e("; ".join(reasons))})</span></li>')
    for _, m in metrics[metrics["상태"] == REVIEW].iterrows():
        items.append(f"<li><b>{_e(m['metric_id'])} {_e(m['지표'])}</b> "
                     f'<span class="dim">{_e(m["이유"])}</span></li>')
    flagged = records[records["상태"] == REVIEW]
    for _, r in flagged.head(15).iterrows():
        items.append(f"<li><b>{_e(r['항목'])} {_e(r['기준일'])}</b> <span class='dim'>{_e(r['비고'])}</span></li>")
    if len(flagged) > 15:
        items.append(f"<li class='dim'>확인 필요 줄이 {len(flagged) - 15}개 더 있어요 (노트북 2번 칸 표 참고)</li>")
    return f"<ul class='review'>{''.join(items)}</ul>" if items else "<p class='dim'>없음</p>"


def check_table(checked):
    if checked is None or checked.empty:
        return "<p class='dim'>숫자 검증을 하지 않았어요 (Gemini 답이 없음). 노트북 4번 칸 메시지를 확인해 주세요.</p>"
    rows = []
    for _, r in checked.iterrows():
        written = "" if is_blank(r["적은 숫자"]) else f"{r['적은 숫자']} {r['적은 단위']}"
        actual = "" if is_blank(r["실제 값"]) else f"{r['실제 값']:,.2f} {r['적은 단위']}"
        css = {PASS: "ok", FIX: "bad", CHECK: "warn"}[r["판정"]]
        rows.append(f"<tr><td>{_e(r['claim_id'])}</td><td>{_e(written)}</td><td>{_e(actual)}</td>"
                    f"<td><span class='chip {css}'>{_e(r['판정'])}</span></td><td class='dim'>{_e(r['이유'])}</td>"
                    f"<td class='rid'>{_e(r['근거 id'])}</td></tr>")
    return ("<table class='grid small'><thead><tr><th>주장</th><th>적은 숫자</th><th>실제 값</th><th>판정</th>"
            f"<th>이유</th><th>근거 id</th></tr></thead><tbody>{''.join(rows)}</tbody></table>")


def source_table(sources):
    rows = []
    for _, s in sources.iterrows():
        url = s["출처 URL"]
        link = f'<a href="{_e(url)}" target="_blank">원문 열기</a>' if str(url).startswith("http") else _e(url)
        rows.append(f"<tr><td>[{_e(s['source_id'])}]</td><td>{_e(s['출처'])}</td><td>{_e(s['기준일 범위'])}</td>"
                    f"<td>{_e(s['수집 방법'])}</td><td>{_e(s['신뢰도'])}</td><td class='url'>{link}</td></tr>")
    return ("<table class='grid small'><thead><tr><th>번호</th><th>출처</th><th>기준일 범위</th><th>수집 방법</th>"
            f"<th>신뢰도</th><th>원본</th></tr></thead><tbody>{''.join(rows)}</tbody></table>")


def body_text(text, claims):
    if not text.strip():
        return "<p class='dim'>설정 칸 TEXT에 내 문장을 적으면 여기에 나와요.</p>"
    out = []
    for paragraph in text_with_status(text, claims):
        spans = []
        for sentence, status, label in paragraph:
            if status is None:
                spans.append(f"<span>{_e(sentence)}</span>")
            else:
                spans.append(f"<span class='s {STATUS_CLASS[status]}' title='{_e(status)}'>{_e(sentence)}"
                             f"{_sup(label)}</span>")
        out.append(f"<p>{' '.join(spans)}</p>")
    return "".join(out)


def kpis(records, metrics, stock_code, shown, kind):
    """Four headline numbers for the latest shown period."""
    latest = shown[-1]
    tiles = []
    if kind == REPORT:
        for name in ("매출액", "영업이익"):
            row = _record(records, stock_code, latest.item(name), latest.base_date)
            ok = row is not None and row["상태"] != REVIEW and not is_blank(row["값"])
            tiles.append((f"{name} {latest.label}", fmt_value(row["값"], "원") if ok else "확인 필요"))
        for label, names in METRIC_ROWS[1:]:
            hit = metrics[(metrics["기간"] == latest.label) & metrics["지표명"].isin(names)]
            ok = len(hit) and not is_blank(hit.iloc[0]["값"])
            tiles.append((f"{label} {latest.label}", fmt_value(hit.iloc[0]["값"], "%") if ok else "확인 필요"))
    else:
        judged = metrics[metrics["판정"] != ""] if "판정" in metrics else metrics.iloc[0:0]
        if judged.empty:
            judged = metrics[(metrics["기간"] == latest.label) & (metrics["상태"] != REVIEW)]
        for _, m in judged.drop_duplicates("지표명", keep="last").head(4).iterrows():
            value = fmt_value(m["값"], m["단위"]) or "확인 필요"
            verdict = f" · {m['판정']}" if m.get("판정") else ""
            tiles.append((f"{m['지표명']} {m['기간']}", value + verdict))
    return "".join(f"<div class='kpi'><div class='k'>{_e(k)}</div><div class='v'>{_e(v)}</div></div>" for k, v in tiles)


CSS = """
:root{--ink:#1b2a41;--sub:#5a6578;--line:#e3e7ee;--bg:#f6f7f9;--accent:#2f5d8a;
--ok:#2e7d4f;--okbg:#e7f4ec;--bad:#b3261e;--badbg:#fbe9e7;--warn:#9a6200;--warnbg:#fff4dc}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.65 "Pretendard","Apple SD Gothic Neo","Malgun Gothic","Noto Sans KR",sans-serif}
.page{max-width:920px;margin:24px auto;background:#fff;border:1px solid var(--line);border-radius:10px;padding:40px 48px}
header{border-bottom:3px solid var(--ink);padding-bottom:14px;margin-bottom:18px}
.kind{display:inline-block;font-size:12px;letter-spacing:.04em;color:#fff;background:var(--accent);padding:2px 10px;border-radius:999px}
h1{font-size:26px;margin:10px 0 4px}
.meta{color:var(--sub);font-size:13px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:18px 0 6px}
.kpi{border:1px solid var(--line);border-radius:8px;padding:10px 12px;background:#fbfcfd}
.kpi .k{font-size:12px;color:var(--sub)}.kpi .v{font-size:19px;font-weight:700;font-variant-numeric:tabular-nums}
h2{font-size:17px;margin:30px 0 10px;padding-left:10px;border-left:4px solid var(--accent)}
p{margin:0 0 10px}
.s{border-bottom:2px solid transparent;padding-bottom:1px}
.s.ok{border-color:#9fd3b2}.s.bad{background:var(--badbg);border-color:var(--bad)}.s.warn{background:var(--warnbg);border-color:#e7c27a}
sup.src{color:var(--accent);font-size:10px;margin-left:1px}
table{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}
.fin th,.fin td{padding:7px 10px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}
.fin thead th{border-bottom:2px solid var(--ink);font-size:13px}
.fin tbody th{text-align:left;font-weight:600}
.fin tr.metric th,.fin tr.metric td{background:#f8fafc;color:#24364f}
.fin thead th:first-child{text-align:left;color:var(--sub);font-weight:500}
.grid th,.grid td{padding:6px 8px;border-bottom:1px solid var(--line);text-align:left;vertical-align:top}
.grid thead th{border-bottom:2px solid var(--ink);font-size:13px}
.grid td.num{text-align:right}
.small{font-size:12.5px}
.formula{font-size:12px;color:var(--sub)}
.chip{display:inline-block;font-size:11.5px;font-weight:700;padding:1px 8px;border-radius:999px}
.chip.ok{color:var(--ok);background:var(--okbg)}.chip.bad{color:var(--bad);background:var(--badbg)}.chip.warn{color:var(--warn);background:var(--warnbg)}
.need{font-size:12px;color:var(--warn);background:var(--warnbg);padding:1px 6px;border-radius:4px;cursor:help}
.dim{color:var(--sub)}
.nw,.chip,.need{white-space:nowrap}
.fin{margin-bottom:14px}
.legend{font-size:12px;color:var(--sub);margin-top:6px}
.legend span{margin-right:12px}
.legend i{display:inline-block;width:18px;height:3px;vertical-align:middle;margin-right:4px}
.chart{width:100%;height:auto;border:1px solid var(--line);border-radius:8px;background:#fff}
.chart .ct{font-size:13px;font-weight:700;fill:var(--ink)}.chart .gl{stroke:#edf0f4}
.chart .yt{font-size:11px;fill:var(--sub);text-anchor:end}.chart .xt{font-size:11px;fill:var(--sub);text-anchor:middle}.chart .lg{font-size:11.5px;fill:var(--ink)}
ul.review{padding-left:18px;margin:0}ul.review li{margin-bottom:6px}
.url{word-break:break-all}
.grid td:first-child,.grid th{white-space:nowrap}.rid{font-size:11px;color:var(--sub);word-break:break-all}
a{color:var(--accent)}
footer{margin-top:32px;padding-top:12px;border-top:1px solid var(--line);font-size:12px;color:var(--sub)}
@media print{body{background:#fff}.page{border:0;margin:0;padding:0}}
"""


def render_draft(kind, title, author, stock_code, stock_name, text, records, sources, metrics, checked,
                 fetched, shown, mine_title, mine, chart_svg, caveats=(), mine_latest=6):
    """One self-contained HTML page. Every number on it comes from the tables passed in."""
    if kind not in (REPORT, INDICATOR):
        raise CollectError(f"KIND는 '{REPORT}' 또는 '{INDICATOR}' 중 하나예요: {kind}")
    claims = summarize_claims(checked)
    counts = {s: sum(1 for c in claims if c[2] == s) for s in STATUS_RANK}
    tally = (" · ".join(f"{s} {n}" for s, n in counts.items() if n) or "검증 안 함")
    text_heading = "투자포인트" if kind == REPORT else "결론"
    legend = ("<div class='legend'><span><i style='background:#9fd3b2'></i>숫자 검증됨</span>"
              "<span><i style='background:var(--bad)'></i>숫자 틀림</span>"
              "<span><i style='background:#e7c27a'></i>확인 필요</span><span>밑줄 없음: 숫자 없는 문장</span></div>")
    judged = "판정" in metrics.columns
    sections = {
        "text": f"<h2>{text_heading}</h2>{body_text(text, claims)}{legend if claims else ''}",
        "fin": f"<h2>실적 표</h2>{financial_tables(records, metrics, stock_code, fetched, shown)}",
        "mine": f"<h2>{_e(mine_title)}</h2>{mine_table(mine, mine_latest)}",
        "chart": f"<h2>차트</h2>{chart_svg}" if chart_svg else "",
        "metrics": f"<h2>지표 표</h2>{metric_table(metrics, judged)}",
        "check": f"<h2>숫자 검증 결과</h2><p class='dim'>주장 {len(claims)}개: {_e(tally)}</p>{check_table(checked)}",
        "review": f"<h2>확인 필요</h2>{review_list(claims, records, metrics, caveats)}",
        "sources": f"<h2>출처</h2>{source_table(sources)}",
    }
    order = (("text", "fin", "mine", "chart", "metrics", "check", "review", "sources") if kind == REPORT
             else ("text", "metrics", "chart", "mine", "fin", "check", "review", "sources"))
    period = f"{shown[0].label} ~ {shown[-1].label}" if len(shown) > 1 else shown[0].label
    return f"""<!DOCTYPE html>
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{_e(title)}</title><style>{CSS}</style></head>
<body><div class="page">
<header><span class="kind">{_e(kind)}</span>
<h1>{_e(title)}</h1>
<div class="meta">{_e(stock_name)} ({_e(stock_code)}) · 대상 기간 {_e(period)} · 작성 {_e(author)} · 생성 {_e(now_kst())}</div>
<div class="kpis">{kpis(records, metrics, stock_code, shown, kind)}</div></header>
{"".join(sections[k] for k in order)}
<footer>KFA AI 에이전트 스터디 2조 · 숫자는 공통 엔진(수집 → rid·[S#] → 지표 [M#] → 숫자 대조)을 거쳤어요.
위첨자 [S#]는 출처 번호, [M#]는 지표 번호예요. 투자 권유가 아니에요.</footer>
</div></body></html>"""


def show_draft(page, path, height=1500):
    """Save the page and show it inside the notebook in an isolated frame."""
    from IPython.display import HTML, display

    path = Path(path)
    path.write_text(page, encoding="utf-8")
    frame = (f'<iframe srcdoc="{html.escape(page, quote=True)}" '
             f'style="width:100%;height:{height}px;border:1px solid #ddd;border-radius:8px"></iframe>')
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        display(HTML(frame))
    print(f"💾 저장: {path}  (왼쪽 📁 파일 창에서 오른쪽 클릭 → 다운로드. 브라우저로 열면 인쇄 → PDF 저장도 돼요)")
    return path

print("✅ 초안 도구 준비 완료")

✅ 초안 도구 준비 완료


In [30]:
DART_API_KEY = get_key("DART_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")
print("DART 키:", "등록됨 ✅" if DART_API_KEY else "없음 ⏭ (Secrets에 DART_API_KEY + 노트북 액세스 켜기)")
print("Gemini 키:", "등록됨 ✅" if GEMINI_API_KEY else "없음 ⏭ (없으면 기본 설정일 때만 미리 받아 둔 답을 써요)")

DART 키: 등록됨 ✅
Gemini 키: 등록됨 ✅


## 1. DART에서 여러 기간 받기

`YEARS`와 `QUARTERS`의 기간마다 **연결** 재무제표 6개 계정을 받아요. 각 기간의 1년 전도 같이 받아요(성장률 계산용).
한 기간이 ❌여도 멈추지 않고 나머지를 계속 받아요. 아직 공시 전인 분기면 `DART 013`이 나와요.

In [31]:
dart, shown, fetched, stock_name = None, [], [], STOCK_CODE
try:
    if KIND not in (REPORT, INDICATOR):
        raise CollectError(f"KIND는 '{REPORT}' 또는 '{INDICATOR}' 중 하나예요: {KIND}")
    shown, fetched = plan_periods(YEARS, QUARTERS)
except CollectError as error:
    print("❌", error)
else:
    print("페이지에 보일 기간:", ", ".join(p.label for p in shown))
    dart = collect_periods(STOCK_CODE, fetched, DART_API_KEY)
if dart is not None:
    names = [n for n in dart["종목명"].dropna().unique() if n]
    stock_name = names[0] if names else STOCK_CODE
    display(dart[["종목명", "항목", "값", "단위", "기준일", "공시일", "상태"]])

페이지에 보일 기간: 2025, 2026 1Q, 2026 2Q, 2026 3Q
✅ DART 2024 사업보고서: 6줄 (확인 필요 0줄)
✅ DART 2025 1Q 1분기보고서: 6줄 (확인 필요 0줄)
✅ DART 2025 2Q 반기보고서: 9줄 (확인 필요 0줄)
✅ DART 2025 3Q 3분기보고서: 9줄 (확인 필요 0줄)
✅ DART 2025 사업보고서: 6줄 (확인 필요 0줄)
✅ DART 2026 1Q 1분기보고서: 6줄 (확인 필요 0줄)
✅ DART 2026 2Q 반기보고서: 9줄 (확인 필요 0줄)
✅ DART 2026 3Q 3분기보고서: 9줄 (확인 필요 0줄)


,종목명,항목,값,단위,기준일,공시일,상태
0,현대자동차,자산총계,339798429000000,원,2024-12-31,2025-03-12,형식 통과
1,현대자동차,자본총계,120275933000000,원,2024-12-31,2025-03-12,형식 통과
2,현대자동차,부채총계,219522496000000,원,2024-12-31,2025-03-12,형식 통과
3,현대자동차,영업이익(연간),14239592000000,원,2024-12-31,2025-03-12,형식 통과
4,현대자동차,당기순이익(연간),13229908000000,원,2024-12-31,2025-03-12,형식 통과
5,현대자동차,매출액(연간),175231153000000,원,2024-12-31,2025-03-12,형식 통과
6,현대자동차,자산총계,343630337000000,원,2025-03-31,2025-05-15,형식 통과
7,현대자동차,자본총계,121565194000000,원,2025-03-31,2025-05-15,형식 통과
8,현대자동차,부채총계,222065143000000,원,2025-03-31,2025-05-15,형식 통과
9,현대자동차,영업이익(3개월),3633609000000,원,2025-03-31,2025-05-15,형식 통과


## 1-1. 내 소스 붙이기

3회차에 만든 `my_records(code)`를 아래 `def my_records(code):` 자리에 **통째로** 붙여 넣어요(4회차 1-1 칸과 같은 방법).
`MY_TARGETS`에는 그 함수에 넣을 종목코드를 적어요. 환율·PPI처럼 종목과 상관없는 소스면 `[STOCK_CODE]` 그대로 두면 돼요.
`MY_LEVEL`은 내 소스의 수집 방법 단계(1 공식 API, 2 라이브러리, 3 파일, 4 크롤링, 5 수동)예요.
비워 두면 ⏭로 넘어가요.

In [32]:
MY_LEVEL = 1
MY_TARGETS = [STOCK_CODE]

def my_records(code):
    # 환율은 종목과 상관없어서 code는 쓰지 않고, 원/달러(0000001)만 받아요
    key = get_key("ECOS_API_KEY")
    url = (f"https://ecos.bok.or.kr/api/StatisticSearch/{key}/json/kr/1/10000/"
           "731Y001/D/20250101/20261231/0000001")
    table = read_json_table(url, "StatisticSearch.row", params={})

    # 분기마다 마지막 영업일 값 = 기말환율 (아직 안 끝난 이번 분기는 제외)
    table["날짜"] = pd.to_datetime(table["TIME"], format="%Y%m%d")
    table["DATA_VALUE"] = pd.to_numeric(table["DATA_VALUE"].astype(str).str.replace(",", ""), errors="coerce")
    table = table.dropna(subset=["DATA_VALUE"]).sort_values("날짜")
    분기 = table["날짜"].dt.to_period("Q")
    table = table[분기 < pd.Timestamp.today().to_period("Q")]
    table = table.groupby(table["날짜"].dt.to_period("Q")).tail(1)[["날짜", "DATA_VALUE"]]

    return collect_table(
        table,
        {"DATA_VALUE": ("원/달러 기말환율", "원")},
        date_col="날짜",
        level=MY_LEVEL,
        source="한국은행 ECOS 주요국 통화의 대원화환율(731Y001), 원/미국달러 매매기준율",
        source_url="https://ecos.bok.or.kr/#/Short/2bd68f",
        reliability="상",
        basis=NOT_APPLICABLE,
        disclosed=SAME_AS_DATE)

mine_parts = []
for code in MY_TARGETS:
    result = run_step(f"내 소스 {code}", MY_LEVEL, lambda: my_records(code))
    if result is not None:
        mine_parts.append(result)
print(f"✅ 내 소스 {sum(len(f) for f in mine_parts)}줄" if mine_parts else "⏭ 내 소스 없음")

✅ 내 소스 005380: 7줄 (확인 필요 0줄)
✅ 내 소스 7줄


## 1-2. 손으로 옮긴 숫자 (선택)

IR 실적발표 자료, 증권사·포털 화면처럼 코드로 못 받는 숫자는 여기에 **한 줄에 하나씩** 적어요. 공통 표에 들어가야 [S#]가 붙고 숫자 대조가 돼요.

```python
MANUAL_ROWS = [
    {"항목": "리테일 고객자산", "값": 315, "단위": "조원", "기준일": "2026-09-30",
     "출처": "삼성증권 3Q26 실적발표", "출처 URL": "https://...", "신뢰도": "상", "확인": "o"},
    {"종목코드": "035720", "종목명": "카카오", "항목": "PER", "값": 31.2, "단위": "배", "기준일": "2026-11-20",
     "출처": "네이버 금융", "출처 URL": "https://finance.naver.com/item/main.naver?code=035720", "신뢰도": "중", "확인": "o"},
]
```

- 필수 칸: `항목` `값` `단위` `기준일` `출처` `출처 URL` `신뢰도`(상·중·하). 다른 종목 숫자(peer PER 등)면 `종목코드`·`종목명`도 적어요.
- `"확인": "o"`는 **원문 화면과 한 번 대조했다**는 표시예요. 없으면 그 줄은 '확인 필요'로 남고, 그 숫자를 쓴 문장은 REVIEW가 나와요.
- 자료 원문(PDF 등)은 레포에 올리지 않아요. 숫자와 출처 주소만 적어요.

In [33]:
MANUAL_ROWS = []

manual = None
if MANUAL_ROWS:
    manual = run_step("손으로 옮긴 숫자", 5, lambda: manual_records(MANUAL_ROWS, STOCK_CODE, stock_name))
else:
    print("⏭ 손으로 옮긴 숫자 없음")

⏭ 손으로 옮긴 숫자 없음


## 2. 줄 이름표(rid)와 출처 번호([S#])

DART·내 소스·손으로 옮긴 숫자를 한 표로 합치고 이름표를 붙여요. 아래 '확인 필요' 줄은 페이지에 값 대신 `확인 필요`로 보여요.

In [34]:
sources, tagged = None, None
parts = [f for f in (dart, *mine_parts, manual) if f is not None]
if dart is None:
    print("⏭ DART 결과가 없어서 건너뛰어요 (1번 칸의 ❌ 메모를 먼저 확인)")
else:
    try:
        sources, tagged = build_sources(with_ids(pd.concat(parts, ignore_index=True)))
    except CollectError as error:
        print("❌", error)
    else:
        display(sources)
        flagged = tagged[tagged["상태"] == REVIEW]
        print(f"확인 필요 줄 {len(flagged)}개")
        if len(flagged):
            print(flagged[["항목", "기준일", "비고"]].to_string())

,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31,6
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-03-31,6
2,S3,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-06-30,9
3,S4,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-09-30,9
4,S5,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-12-31,6
5,S6,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2026-03-31,6
6,S7,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2026-06-30,9
7,S8,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2026-09-30,9
8,S9,"한국은행 ECOS 주요국 통화의 대원화환율(731Y001), 원/미국달러 매매기준율",https://ecos.bok.or.kr/#/Short/2bd68f,1 공식 API,상,2025-03-31 ~ 2026-09-30,7


확인 필요 줄 0개


## 3. 지표 [M#]

기본 지표는 기간마다 계산해요. 연간: 매출 성장률·영업이익률·부채비율 / 분기: 매출 성장률(전년 동기)·영업이익률(분기)·부채비율.

**내 지표 더하기 (선택).** 공통 표에 있는 항목 두 개로 만드는 비율(`"ratio"`)이나 1년 전 대비 증가율(`"growth"`)을 더할 수 있어요. 연간 기간에만 계산돼요.
입력 항목은 **기준이 '연결'인 줄**만 써요(DART 줄, 또는 `기준` 칸을 `"연결"`로 적은 내 소스 줄).

```python
EXTRA_METRICS = [
    MetricSpec("ROE", "%", "당기순이익(연간) ÷ 자본총계 × 100", "ratio", ("당기순이익(연간)", "자본총계")),
]
```

**기준과 비교하기 (지표분석형).** `BENCHMARKS`에 지표 이름과 (`"이상"` 또는 `"이하"`, 숫자)를 적으면 코드가 충족/미달을 판정해요.

```python
BENCHMARKS = {"부채비율": ("이하", 200), "ROE": ("이상", 10)}
```

In [35]:
EXTRA_METRICS = []
BENCHMARKS = {}

In [36]:
metrics = None
if tagged is None:
    print("⏭ 줄 이름표가 없어서 건너뛰어요")
else:
    try:
        metrics = period_metrics(tagged, STOCK_CODE, shown, EXTRA_METRICS)
        if BENCHMARKS:
            metrics = judge_metrics(metrics, BENCHMARKS)
    except CollectError as error:
        print("❌", error)
    else:
        columns = ["metric_id", "지표", "값", "단위", "상태", "이유"] + (["비교 기준", "판정"] if BENCHMARKS else [])
        display(metrics[columns])

,metric_id,지표,값,단위,상태,이유
0,M1,매출 성장률 (2025),6.29073,%,계산됨,
1,M2,영업이익률 (2025),6.157088,%,계산됨,
2,M3,부채비율 (2025),188.954127,%,계산됨,
3,M4,매출 성장률(전년 동기) (2026 1Q),3.447877,%,계산됨,
4,M5,영업이익률(분기) (2026 1Q),5.473981,%,계산됨,
5,M6,부채비율 (2026 1Q),190.284871,%,계산됨,
6,M7,매출 성장률(전년 동기) (2026 2Q),1.923203,%,계산됨,
7,M8,영업이익률(분기) (2026 2Q),5.792683,%,계산됨,
8,M9,부채비율 (2026 2Q),191.332539,%,계산됨,
9,M10,매출 성장률(전년 동기) (2026 3Q),-3.448525,%,계산됨,


## 4. Gemini에게 내 문장 보내기

내 문장(`TEXT`)과 위 두 표를 보내고, 문장 속 숫자마다 근거 줄을 고르게 해요. 답은 **받은 그대로** 파일에 저장해요.
같은 질문으로 다시 돌리면 Gemini를 또 부르지 않아요(♻). 한 번 실행에 Gemini는 **1번**만 불러요(무료 키 하루 20번).

In [37]:
GEMINI_MODEL = "gemini-3.8-flash"
GEMINI_FALLBACK_MODEL = "gemini-3.5-flash-lite"
AI_FILE = f"w06_ai_claims_{STOCK_CODE}.txt"

claims_path = None
if metrics is None:
    print("⏭ 지표가 없어서 건너뛰어요")
elif not TEXT.strip():
    print("⏭ TEXT가 비어 있어서 숫자 검증을 건너뛰어요. 페이지는 표와 차트만으로 만들어요")
else:
    prompt = claims_prompt(tagged, metrics, TEXT)
    try:
        claims_path = ask_claims(prompt, GEMINI_API_KEY, AI_FILE, GEMINI_MODEL, GEMINI_FALLBACK_MODEL)
    except (CollectError, SkipStep) as error:
        print("⚠", error)
        claims_path = use_backup(prompt, AI_FILE, f"ai_claims_{STOCK_CODE}_{max(YEARS) if YEARS else 0}")

⏳ Gemini 요청 한도에 걸렸어요 (429). 5초 기다렸다가 다시 시도해요 (1/3)
⏳ Gemini 요청 한도에 걸렸어요 (429). 10초 기다렸다가 다시 시도해요 (2/3)
⏳ Gemini 요청 한도에 걸렸어요 (429). 20초 기다렸다가 다시 시도해요 (3/3)
⏳ 계속 안 돼서 예비 모델(gemini-3.5-flash-lite)로 한 번 더 시도해요
⚠ Gemini 서버가 계속 붐비거나 오류가 나요 (503). 내 설정 문제가 아니에요. 잠시 뒤 다시 실행해 주세요
   원문: gemini-3.5-flash-lite 503: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
⏭ 미리 받아 둔 답이 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요


## 5. 숫자 대조

Gemini 답을 원문 그대로 보여 주고, 문장에 적힌 숫자를 근거와 같은 단위로 맞춰 비교해요.
- **PASS** 맞음 · **FIX** 틀림(실제 값을 보여 줘요) · **REVIEW** 사람이 봐야 함(근거 없음, 단위가 원·%·같은 글자가 아님 등)

In [38]:
checked = None
if claims_path is None:
    print("⏭ 저장한 답이 없어서 건너뛰어요")
else:
    print(Path(claims_path).read_text(encoding="utf-8"))
    checked = check_claims(load_claims(claims_path), metrics, tagged)
    display(checked)

⏭ 저장한 답이 없어서 건너뛰어요


In [39]:
# Gemini 근거가 표에서 안 찾아지면, 사람이 고른 근거로 대조 (판정은 똑같이 check_claims)
import pandas as pd

# ── 사람이 고른 근거: TEXT 문장을 바꾸면 여기도 같이 고쳐요 ──
#    지표 번호(M1·M2·M3)는 초안 페이지 2025 연간 칸의 번호. 기간·내 지표를 바꾸면 다시 확인
def manual_claim_list():
    return [
        {"claim_id": "C-1", "문장": "현대차의 2025년 매출은 186.3조 원으로 전년보다 6.3% 늘었다.",
         "근거": [(rid("매출액(연간)", "2025-12-31"), "186.3", "조 원"),
                  ("M1", "6.3", "%")]},
        {"claim_id": "C-2", "문장": "2025년 영업이익률은 6.2%였다.",
         "근거": [("M2", "6.2", "%")]},
        {"claim_id": "C-3", "문장": "부채비율은 189% 수준이다.",
         "근거": [("M3", "189", "%")]},
        {"claim_id": "C-4", "문장": "2026년 3분기 영업이익은 2.7조 원이었다.",
         "근거": [(rid("영업이익(3개월)", "2026-09-30"), "2.7", "조 원")]},
        {"claim_id": "C-5", "문장": "원/달러 기말환율은 2026년 6월 말 1,541.5원에서 9월 말 1,358.4원으로 떨어졌다.",
         "근거": [(rid("원/달러 기말환율", "2026-06-30"), "1,541.5", "원"),
                  (rid("원/달러 기말환율", "2026-09-30"), "1,358.4", "원")]},
    ]

# ── 표 찾기와 rid 찾기 (rid는 길어서 손으로 옮기지 않고 코드가 찾아요) ──
def _find_table(names, must_have):
    for n in names:
        v = globals().get(n)
        if isinstance(v, pd.DataFrame) and must_have <= set(v.columns):
            return n, v
    for n, v in globals().items():
        if not n.startswith("_") and isinstance(v, pd.DataFrame) and must_have <= set(v.columns):
            return n, v
    raise NameError(f"{must_have} 칸이 있는 표를 못 찾았어요")

tbl_name, tagged = _find_table(["tagged", "all_tagged", "records_tagged", "dart_tagged"], {"rid", "항목", "기준일"})
met_name, mets = _find_table(["metrics"], {"metric_id", "지표"})
print(f"공통 표: {tbl_name} ({len(tagged)}줄) / 지표 표: {met_name} ({len(mets)}줄)")

def rid(항목, 기준일):
    hit = tagged[(tagged["항목"] == 항목) & (tagged["기준일"].astype(str).str[:10] == 기준일)]
    if len(hit) != 1:
        raise ValueError(f"'{항목}' {기준일} 줄이 {len(hit)}개예요")
    return hit["rid"].iloc[0]

# ── 사람 근거로 바꿀지 판단: 결과만 보고 정해요 ──
gemini_checked = globals().get("checked")
reasons = []
if globals().get("claims") is None or gemini_checked is None:
    reasons.append("Gemini 답 없음")
else:
    missing = gemini_checked["이유"].astype(str).str.contains("공통 표에서 못 찾음")
    if missing.any():
        reasons.append(f"Gemini 근거 rid를 공통 표에서 못 찾음 {int(missing.sum())}개")

if reasons:
    checked = check_claims(manual_claim_list(), mets, tagged)
    note = "숫자 대조 근거는 사람이 직접 고름 (이유: " + ", ".join(reasons) + "). 판정은 코드(check_claims)"
    if note not in CAVEATS:
        CAVEATS.append(note)
    print("👤 사람이 고른 근거로 대조했어요:", ", ".join(reasons))
else:
    print("🤖 Gemini 근거를 그대로 써요")
display(checked)

공통 표: tagged (67줄) / 지표 표: metrics (12줄)
👤 사람이 고른 근거로 대조했어요: Gemini 답 없음


,claim_id,문장,근거 id,적은 숫자,적은 단위,실제 값,판정,이유,출처 번호
0,C-1,현대차의 2025년 매출은 186.3조 원으로 전년보다 6.3% 늘었다.,005380|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무제표...,186.3,조 원,186.254472,PASS,,[S1][S5]
1,C-1,현대차의 2025년 매출은 186.3조 원으로 전년보다 6.3% 늘었다.,M1,6.3,%,6.290730,PASS,,[S1][S5]
2,C-2,2025년 영업이익률은 6.2%였다.,M2,6.2,%,6.157088,PASS,,[S5]
3,C-3,부채비율은 189% 수준이다.,M3,189,%,188.954127,PASS,,[S5]
4,C-4,2026년 3분기 영업이익은 2.7조 원이었다.,005380|영업이익(3개월)|연결|2026-09-30|DART 단일회사 전체 재무...,2.7,조 원,2.677232,PASS,,[S8]
5,C-5,"원/달러 기말환율은 2026년 6월 말 1,541.5원에서 9월 말 1,358.4원...",|원/달러 기말환율|해당 없음|2026-06-30|한국은행 ECOS 주요국 통화의 ...,"1,541.5",원,1541.500000,PASS,,[S9]
6,C-5,"원/달러 기말환율은 2026년 6월 말 1,541.5원에서 9월 말 1,358.4원...",|원/달러 기말환율|해당 없음|2026-09-30|한국은행 ECOS 주요국 통화의 ...,"1,358.4",원,1358.400000,PASS,,[S9]


## 6. 초안 페이지

위 결과로 HTML 한 장을 만들어 아래에 보여 주고 파일로 저장해요. 문장에 그은 밑줄 색은 숫자 검증 결과예요(초록 맞음, 빨강 틀림, 노랑 확인 필요).

In [40]:
page_path = None
if metrics is None:
    print("⏭ 지표가 없어서 페이지를 못 만들어요. 위 칸의 ❌·⏭ 메모부터 확인해 주세요")
else:
    try:
        chart = (item_chart(tagged, CHART_ITEMS, 260, CHART_REBASE) if CHART_ITEMS
                 else default_chart(tagged, STOCK_CODE, fetched))
        mine = tagged[tagged["출처"] != DART_SOURCE]
        page = render_draft(KIND, TITLE, AUTHOR, STOCK_CODE, stock_name, TEXT, tagged, sources, metrics, checked,
                            fetched, shown, MY_TABLE_TITLE, mine, chart, CAVEATS)
        page_path = show_draft(page, f"w06_초안_{STOCK_CODE}.html")
    except CollectError as error:
        print("❌", error)

💾 저장: w06_초안_005380.html  (왼쪽 📁 파일 창에서 오른쪽 클릭 → 다운로드. 브라우저로 열면 인쇄 → PDF 저장도 돼요)


> 💾 Colab 런타임이 끝나면 파일이 사라져요. **HTML은 왼쪽 📁 파일 창에서 내려받고**, 노트북은 **파일 → GitHub에 사본 저장**으로 내 레포 `kfa-ai-agent`에 `w06_초안.ipynb`로 올려요.
> 레포는 Public이라 키 값이나 자료 원문(PDF)은 올리지 않아요.

In [41]:
# 초안 HTML 내려받기 + 새 탭에서 열어 인쇄 (2조 추가 셀, 원본 노트북에는 없음)
import base64
from google.colab import files
from IPython.display import HTML, display

files.download(str(page_path))
b64 = base64.b64encode(open(page_path, "rb").read()).decode()
display(HTML(f"""<button id="pp" style="font-size:16px;padding:8px 16px">새 탭에서 열기 + 인쇄</button>
<script>
document.getElementById("pp").onclick = function () {{
  const bytes = Uint8Array.from(atob("{b64}"), c => c.charCodeAt(0));
  const page = new TextDecoder("utf-8").decode(bytes);
  const w = window.open("", "_blank");
  w.document.open();
  w.document.write(page);
  w.document.close();
  w.onload = function () {{ w.print(); }};
  setTimeout(function () {{ w.print(); }}, 800);
}};
</script>"""))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>